In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from rdkit import Chem

In [2]:
# すべてのデータセットを読み込み
datasets = {
    'Buchwald-Hartwig': '../data/Buchwald-Hartwig/Dreher_and_Doyle_reaction_t5_ready.csv',
    'NiB': '../data/NiB/inchi_23l_reaction_t5_ready.csv',
    'Suzuki-Miyaura': '../data/Suzuki-Miyaura/aap9112_reaction_t5_ready.csv',
    'ORD': '../data/ORD/all_ord_reaction_uniq_with_attr20240506_v3_train.csv'
    
}

dfs = {}
for name, path in datasets.items():
    try:
        dfs[name] = pd.read_csv(path)
        print(f"Loaded {name}: {len(dfs[name])} rows")
        
        # ORDデータセットの場合、重要なカラムに欠損値がある行を削除
        if name == 'ORD':
            required_cols = ['REACTANT', 'PRODUCT']
            before_count = len(dfs[name])
            dfs[name] = dfs[name].dropna(subset=required_cols)
            after_count = len(dfs[name])
            print(f"  Dropped {before_count - after_count} rows with missing values in required columns")
            print(f"  Remaining: {after_count} rows")
        
        # CATALYST、REAGENT、SOLVENTをREAGENTカラムに結合
        dfs[name]["REAGENT"] = dfs[name]["CATALYST"].fillna(" ") + "." + dfs[name]["REAGENT"].fillna(" ") + "." + dfs[name]["SOLVENT"].fillna(" ")
        
    except Exception as e:
        print(f"Error loading {name}: {e}")

Loaded Buchwald-Hartwig: 1980 rows
Loaded NiB: 1518 rows
Loaded Suzuki-Miyaura: 3696 rows
Loaded ORD: 1819521 rows
  Dropped 18172 rows with missing values in required columns
  Remaining: 1801349 rows


In [3]:
from pathlib import Path

# SMILESをcanonical化する関数
def canonicalize_smiles(smiles_str):
    """RDKitを使ってSMILESをcanonical形式に変換。canonical化できない場合はNoneを返す。"""
    if pd.isna(smiles_str) or smiles_str == '':
        return ''
    
    # '.'で区切られた複数のSMILESを処理
    smiles_list = str(smiles_str).split('.')
    canonical_list = []
    
    for smi in smiles_list:
        smi = smi.strip()
        if smi == '' or smi == ' ':
            continue
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                canonical_list.append(Chem.MolToSmiles(mol))
            else:
                # パースに失敗した場合、削除対象としてNoneを返す
                return None
        except:
            # エラーが発生した場合、削除対象としてNoneを返す
            return None
    
    # ソートして結合し、一貫した順序を保証
    return '.'.join(sorted(canonical_list))

# DataFrameの全SMILES列をcanonical化する関数
def canonicalize_dataframe(df, smiles_cols=['REACTANT', 'REAGENT', 'PRODUCT']):
    """DataFrameのSMILES列をcanonical化し、無効なSMILESを含む行を削除。
    canonical化されたSMILESを持つ新しいDataFrameを返す。"""
    
    # 利用可能な列をチェック
    available_cols = [col for col in smiles_cols if col in df.columns]
    if not available_cols:
        print(f"Warning: None of the specified columns {smiles_cols} found in dataframe")
        return df
    
    print(f"Canonicalizing SMILES columns: {available_cols}")
    
    # オリジナルを変更しないようにコピーを作成
    canonical_df = df.copy()
    
    # 各列をcanonical化
    for col in available_cols:
        print(f"  Processing {col}...")
        canonical_df[col] = canonical_df[col].apply(canonicalize_smiles)
    
    # None値（canonical化失敗）を含む行をカウントして削除
    none_mask = canonical_df[available_cols].isnull().any(axis=1)
    none_count = none_mask.sum()
    total_count = len(canonical_df)
    
    if none_count > 0:
        print(f"  無効なSMILESを含む{none_count}行を削除（全{total_count}行中）")
        canonical_df = canonical_df[~none_mask]
        print(f"  残り: {len(canonical_df)}行")
    else:
        print(f"  全{total_count}行が有効なSMILESです")
    
    return canonical_df

# canonical化されたCSVファイルのパスを取得する関数
def get_canonical_path(name, original_path):
    """データセット名と元のパスから、canonical化されたCSVファイルのパスを生成"""
    original_path = Path(original_path)
    dataset_dir = original_path.parent
    stem = original_path.stem.replace('_reaction_t5_ready', '')
    canonical_filename = f"{stem}_canonicalized.csv"
    return dataset_dir / canonical_filename

# マッチング用のユニークキーを作成する関数
def create_match_key(canonical_df, key_cols=['REACTANT', 'REAGENT', 'PRODUCT']):
    """指定された列を結合してユニークキーを作成。
    DataFrameは既にcanonical化されていることを前提とする。"""
    
    available_cols = [col for col in key_cols if col in canonical_df.columns]
    if len(available_cols) != len(key_cols):
        print(f"Warning: Missing columns {set(key_cols) - set(available_cols)}")
        return None
    
    # マッチキーを作成
    match_keys = canonical_df[available_cols].apply(
        lambda row: '||'.join(row.astype(str)), axis=1
    )
    return match_keys

# 全データセットをcanonical化（キャッシュを利用）
canonical_dfs = {}
for name, df in dfs.items():
    canonical_path = get_canonical_path(name, datasets[name])
    
    if canonical_path.exists():
        print(f"\n{name}: Loading from cached file {canonical_path}")
        canonical_dfs[name] = pd.read_csv(canonical_path)
        # REAGENT列が存在しない場合は再結合
        if 'REAGENT' not in canonical_dfs[name].columns:
            canonical_dfs[name]["REAGENT"] = (
                canonical_dfs[name]["CATALYST"].fillna(" ") + "." +
                canonical_dfs[name]["REAGENT"].fillna(" ") + "." +
                canonical_dfs[name]["SOLVENT"].fillna(" ")
            )
        print(f"  Loaded {len(canonical_dfs[name])} rows")
    else:
        print(f"\n{name}: Canonicalizing and saving to {canonical_path}")
        canonical_dfs[name] = canonicalize_dataframe(df)
        canonical_path.parent.mkdir(parents=True, exist_ok=True)
        canonical_dfs[name].to_csv(canonical_path, index=False)
        print(f"  Saved {len(canonical_dfs[name])} rows to {canonical_path}")

# 各データセットのマッチキーを作成
match_keys = {}
for name, canonical_df in canonical_dfs.items():
    print(f"\nCreating match keys for {name}...")
    keys = create_match_key(canonical_df)
    if keys is not None:
        match_keys[name] = set(keys)
        print(f"{name}: {len(match_keys[name])} unique combinations")
    else:
        print(f"{name}: Cannot create match keys due to missing columns")


Buchwald-Hartwig: Loading from cached file ../data/Buchwald-Hartwig/Dreher_and_Doyle_canonicalized.csv
  Loaded 1980 rows

NiB: Loading from cached file ../data/NiB/inchi_23l_canonicalized.csv
  Loaded 1518 rows

Suzuki-Miyaura: Loading from cached file ../data/Suzuki-Miyaura/aap9112_canonicalized.csv
  Loaded 3696 rows

ORD: Loading from cached file ../data/ORD/all_ord_reaction_uniq_with_attr20240506_v3_train_canonicalized.csv
  Loaded 545244 rows

Creating match keys for Buchwald-Hartwig...
Buchwald-Hartwig: 1980 unique combinations

Creating match keys for NiB...
NiB: 1518 unique combinations

Creating match keys for Suzuki-Miyaura...
Suzuki-Miyaura: 3696 unique combinations

Creating match keys for ORD...
ORD: 358702 unique combinations


In [4]:
# 各データセットのマッチキーを作成
match_keys = {}
for name, canonical_df in canonical_dfs.items():
    print(f"\nCreating match keys for {name}...")
    keys = create_match_key(canonical_df, key_cols=['REACTANT', 'REAGENT', 'PRODUCT'])
    if keys is not None:
        match_keys[name] = set(keys)
        print(f"{name}: {len(match_keys[name])} unique combinations")
    else:
        print(f"{name}: Cannot create match keys due to missing columns")


# 各反応データセットとORDのマッチを計算
if 'ORD' in match_keys:
    ord_keys = match_keys['ORD']
    print(f"ORD dataset has {len(ord_keys)} unique combinations\n")
    
    results = {}
    for name in ['Buchwald-Hartwig', 'NiB', 'Suzuki-Miyaura']:
        if name in match_keys:
            dataset_keys = match_keys[name]
            matches = dataset_keys.intersection(ord_keys)
            match_count = len(matches)
            total_count = len(dataset_keys)
            match_percentage = (match_count / total_count) * 100 if total_count > 0 else 0
            
            results[name] = {
                'total': total_count,
                'matches': match_count,
                'percentage': match_percentage
            }
            
            print(f"=== {name} vs ORD ===")
            print(f"Total unique combinations in {name}: {total_count}")
            print(f"Matches with ORD: {match_count}")
            print(f"Match percentage: {match_percentage:.2f}%")
            print()
else:
    print("ORD dataset not available for comparison")


Creating match keys for Buchwald-Hartwig...
Buchwald-Hartwig: 1980 unique combinations

Creating match keys for NiB...
NiB: 1518 unique combinations

Creating match keys for Suzuki-Miyaura...
Suzuki-Miyaura: 3696 unique combinations

Creating match keys for ORD...
ORD: 358702 unique combinations
ORD dataset has 358702 unique combinations

=== Buchwald-Hartwig vs ORD ===
Total unique combinations in Buchwald-Hartwig: 1980
Matches with ORD: 0
Match percentage: 0.00%

=== NiB vs ORD ===
Total unique combinations in NiB: 1518
Matches with ORD: 0
Match percentage: 0.00%

=== Suzuki-Miyaura vs ORD ===
Total unique combinations in Suzuki-Miyaura: 3696
Matches with ORD: 0
Match percentage: 0.00%



In [5]:
# REACTANT + PRODUCTのみのマッチキーを作成（canonical化されたDataFrameを再利用）
match_keys_reactant_product = {}
for name, canonical_df in canonical_dfs.items():
    print(f"\nCreating match keys for {name} (REACTANT + PRODUCT only)...")
    keys = create_match_key(canonical_df, key_cols=['REACTANT', 'PRODUCT'])
    if keys is not None:
        match_keys_reactant_product[name] = set(keys)
        print(f"{name}: {len(match_keys_reactant_product[name])} unique combinations")
    else:
        print(f"{name}: Cannot create match keys due to missing columns")

# REACTANT + PRODUCTのみに基づいて各反応データセットとORDのマッチを計算
if 'ORD' in match_keys_reactant_product:
    ord_keys = match_keys_reactant_product['ORD']
    print(f"\nORD dataset has {len(ord_keys)} unique REACTANT + PRODUCT combinations\n")
    
    results = {}
    for name in ['Buchwald-Hartwig', 'NiB', 'Suzuki-Miyaura']:
        if name in match_keys_reactant_product:
            dataset_keys = match_keys_reactant_product[name]
            matches = dataset_keys.intersection(ord_keys)
            match_count = len(matches)
            total_count = len(dataset_keys)
            match_percentage = (match_count / total_count) * 100 if total_count > 0 else 0
            
            results[name] = {
                'total': total_count,
                'matches': match_count,
                'percentage': match_percentage
            }
            
            print(f"=== {name} vs ORD ===")
            print(f"Total unique combinations in {name}: {total_count}")
            print(f"Matches with ORD (REACTANT + PRODUCT): {match_count}")
            print(f"Match percentage: {match_percentage:.2f}%")
            print()
else:
    print("ORD dataset not available for comparison")


Creating match keys for Buchwald-Hartwig (REACTANT + PRODUCT only)...
Buchwald-Hartwig: 15 unique combinations

Creating match keys for NiB (REACTANT + PRODUCT only)...
NiB: 33 unique combinations

Creating match keys for Suzuki-Miyaura (REACTANT + PRODUCT only)...
Suzuki-Miyaura: 12 unique combinations

Creating match keys for ORD (REACTANT + PRODUCT only)...
ORD: 344312 unique combinations

ORD dataset has 344312 unique REACTANT + PRODUCT combinations

=== Buchwald-Hartwig vs ORD ===
Total unique combinations in Buchwald-Hartwig: 15
Matches with ORD (REACTANT + PRODUCT): 15
Match percentage: 100.00%

=== NiB vs ORD ===
Total unique combinations in NiB: 33
Matches with ORD (REACTANT + PRODUCT): 0
Match percentage: 0.00%

=== Suzuki-Miyaura vs ORD ===
Total unique combinations in Suzuki-Miyaura: 12
Matches with ORD (REACTANT + PRODUCT): 0
Match percentage: 0.00%



In [6]:
# REAGENTの各SMILES成分がORDにどれだけ含まれているかを分析
# ただし、REACTANT+PRODUCTが一致する反応のみを対象とする

def get_reagent_smiles_set(canonical_df):
    """DataFrameのREAGENT列から個別のSMILESのセットを抽出"""
    all_smiles = set()
    for reagent in canonical_df['REAGENT']:
        if pd.notna(reagent) and reagent != '':
            smiles_list = [s.strip() for s in str(reagent).split('.') if s.strip() and s.strip() != ' ']
            all_smiles.update(smiles_list)
    return all_smiles

def calculate_reagent_match_percentage(reagent_str, ord_smiles_set):
    """
    REAGENT文字列中の各SMILES成分がORDに含まれる割合を計算する。

    Parameters:
    - reagent_str: ドット区切りのSMILES文字列
    - ord_smiles_set: ORDデータセットに含まれる個別のSMILESのセット

    Returns:
    - マッチ割合 (0.0〜1.0)
    """
    if pd.isna(reagent_str) or reagent_str == '':
        return 0.0
    
    smiles_list = [s.strip() for s in str(reagent_str).split('.') if s.strip() and s.strip() != ' ']
    
    if len(smiles_list) == 0:
        return 0.0
    
    matched_count = sum(1 for s in smiles_list if s in ord_smiles_set)
    return matched_count / len(smiles_list)

# ORDデータセットからREAGENTの個別SMILES成分を抽出
print("Extracting REAGENT components from ORD dataset...")
ord_reagent_smiles = get_reagent_smiles_set(canonical_dfs['ORD'])
print(f"Unique REAGENT components in ORD: {len(ord_reagent_smiles)}")

# ORDのREACTANT+PRODUCT -> REAGENTのマッピングを構築
print("\nBuilding ORD REACTANT+PRODUCT mapping...")
ord_df = canonical_dfs['ORD']
ord_reactant_product_to_reagents = defaultdict(list)
for idx, row in ord_df.iterrows():
    key = f"{row['REACTANT']}||{row['PRODUCT']}"
    ord_reactant_product_to_reagents[key].append(row['REAGENT'])

print(f"Unique REACTANT+PRODUCT combinations in ORD: {len(ord_reactant_product_to_reagents)}")

# REACTANT+PRODUCTが一致する反応のみを対象に、各データセットのREAGENTマッチ率を計算
reagent_match_results = {}

for name in ['Buchwald-Hartwig', 'NiB', 'Suzuki-Miyaura']:
    if name not in canonical_dfs:
        continue
    
    print(f"\nCalculating REAGENT match rate for {name}...")
    df = canonical_dfs[name]
    
    match_percentages = []
    matched_reactions = 0
    match_details = []

    component_match_stats = defaultdict(lambda: {'matched': 0, 'total': 0})
    component_unmatch_stats = defaultdict(lambda: {'unmatched': 0, 'total': 0})
    
    for idx, row in df.iterrows():
        reactant_product_key = f"{row['REACTANT']}||{row['PRODUCT']}"
        
        if reactant_product_key in ord_reactant_product_to_reagents:
            matched_reactions += 1
            
            ord_reagents = ord_reactant_product_to_reagents[reactant_product_key]
            
            max_match_pct = 0.0
            best_match_info = None
            
            for ord_reagent in ord_reagents:
                ord_reagent_components = set()
                if pd.notna(ord_reagent) and ord_reagent != '':
                    ord_reagent_components = set(s.strip() for s in str(ord_reagent).split('.')
                                                if s.strip() and s.strip() != ' ')
                
                current_reagent = row['REAGENT']
                if pd.isna(current_reagent) or current_reagent == '':
                    current_components = []
                else:
                    current_components = [s.strip() for s in str(current_reagent).split('.')
                                        if s.strip() and s.strip() != ' ']
                
                if len(current_components) > 0:
                    matched_smiles = [s for s in current_components if s in ord_reagent_components]
                    unmatched_smiles = [s for s in current_components if s not in ord_reagent_components]
                    matched_count = len(matched_smiles)
                    match_pct = matched_count / len(current_components)
                    
                    if match_pct > max_match_pct:
                        max_match_pct = match_pct
                        best_match_info = {
                            'current_reagent': current_reagent,
                            'ord_reagent': ord_reagent,
                            'matched_smiles': matched_smiles,
                            'unmatched_smiles': unmatched_smiles,
                            'total_components': len(current_components),
                            'matched_count': matched_count
                        }
            
            match_percentages.append(max_match_pct * 100)
            
            if best_match_info:
                match_details.append(best_match_info)
                
                for smiles in best_match_info['matched_smiles']:
                    component_match_stats[smiles]['matched'] += 1
                    component_match_stats[smiles]['total'] += 1
                
                for smiles in best_match_info['unmatched_smiles']:
                    component_unmatch_stats[smiles]['unmatched'] += 1
                    component_unmatch_stats[smiles]['total'] += 1
    
    print(f"  Reactions with matching REACTANT+PRODUCT: {matched_reactions} / {len(df)}")
    
    if len(match_percentages) == 0:
        print("  No matching reactions found. Skipping.")
        continue
    
    total_reactions = len(match_percentages)
    
    reagent_match_results[name] = {
        'match_percentages': match_percentages,
        'total_reactions': total_reactions,
        'mean_match': np.mean(match_percentages),
        'median_match': np.median(match_percentages),
        'match_details': match_details,
        'component_match_stats': component_match_stats,
        'component_unmatch_stats': component_unmatch_stats
    }
    
    print(f"\n=== {name}: REAGENT Component Match Analysis ===")
    print(f"Analyzed reactions (REACTANT+PRODUCT matched): {total_reactions}")
    print(f"Mean match rate: {reagent_match_results[name]['mean_match']:.2f}%")
    print(f"Median match rate: {reagent_match_results[name]['median_match']:.2f}%")
    
    # マッチ率をm/n整数比でカウントして表示
    from collections import Counter
    ratio_counter = Counter(
        (d['matched_count'], d['total_components']) for d in match_details
    )
    print("\nMatch rate distribution (matched/total components):")
    for (m, n), count in sorted(ratio_counter.items(), key=lambda x: (x[0][1], x[0][0])):
        print(f"  {m}/{n}: {count} reactions")
    
    print(f"\n=== Matched components ({len(component_match_stats)} total) ===")
    sorted_matched = sorted(component_match_stats.items(),
                           key=lambda x: x[1]['matched'],
                           reverse=True)
    for i, (smiles, stats) in enumerate(sorted_matched, 1):
        print(f"{i:3d}. {smiles}")
        print(f"     {stats['matched']} / {total_reactions} reactions ({stats['matched']/total_reactions*100:.1f}%)")
    
    print(f"\n=== Unmatched components ({len(component_unmatch_stats)} total) ===")
    sorted_unmatched = sorted(component_unmatch_stats.items(),
                             key=lambda x: x[1]['unmatched'],
                             reverse=True)
    for i, (smiles, stats) in enumerate(sorted_unmatched, 1):
        print(f"{i:3d}. {smiles}")
        print(f"     {stats['unmatched']} / {total_reactions} reactions ({stats['unmatched']/total_reactions*100:.1f}%)")

Extracting REAGENT components from ORD dataset...
Unique REAGENT components in ORD: 1740

Building ORD REACTANT+PRODUCT mapping...
Unique REACTANT+PRODUCT combinations in ORD: 344312

Calculating REAGENT match rate for Buchwald-Hartwig...
  Reactions with matching REACTANT+PRODUCT: 1980 / 1980

=== Buchwald-Hartwig: REAGENT Component Match Analysis ===
Analyzed reactions (REACTANT+PRODUCT matched): 1980
Mean match rate: 49.95%
Median match rate: 50.00%

Match rate distribution (matched/total components):
  1/4: 4 reactions
  2/4: 1976 reactions

=== Matched components (25 total) ===
  1. CN(C)C(=NC(C)(C)C)N(C)C
     660 / 1980 reactions (33.3%)
  2. CCN=P(N=P(N(C)C)(N(C)C)N(C)C)(N(C)C)N(C)C
     660 / 1980 reactions (33.3%)
  3. CN1CCCN2CCCN=C12
     660 / 1980 reactions (33.3%)
  4. c1ccc(-c2ccno2)cc1
     90 / 1980 reactions (4.5%)
  5. c1ccc(-c2cnoc2)cc1
     90 / 1980 reactions (4.5%)
  6. c1ccc(-c2ccon2)cc1
     90 / 1980 reactions (4.5%)
  7. c1ccc(CN(Cc2ccccc2)c2ccno2)cc1
     9